# 00. セットアップ — 環境構築・認証・パス設定

このノートブックでは以下を行います:

1. **Google Drive マウント** — Colab 実行時のみ
2. **GitHub リポジトリ取得 / 最新化** — `git clone` または `git pull`
3. **pip install** — 必要パッケージの一括インストール
4. **環境変数チェック** — API キーの確認
5. **パス設定** — `staging/` サブディレクトリの確保

このノートブックを最初に実行してください。以降のノートブック (`01_collect_raw_data.ipynb` 等) はここで定義した変数を前提とします。

In [ ]:
# ── Google Drive マウント（Colab 実行時のみ）──────────────────────────────────
import sys, os

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BASE = "/content/drive/MyDrive/AI_TradeManagement"
else:
    DRIVE_BASE = None  # ローカル実行時は不要

print(f"IN_COLAB={IN_COLAB}, DRIVE_BASE={DRIVE_BASE}")

In [ ]:
# ── GitHub からリポジトリ取得 / 最新化 ──────────────────────────────────────
REPO_URL  = "https://github.com/tsp0918/AI_TradeManagement"  # 実際のURL に変更
REPO_NAME = "AI_TradeManagement"

if IN_COLAB:
    import subprocess
    result = subprocess.run(
        f"git -C /content/{REPO_NAME} pull || git clone {REPO_URL} /content/{REPO_NAME}",
        shell=True, capture_output=True, text=True
    )
    print(result.stdout or result.stderr)
    REPO_BASE = f"/content/{REPO_NAME}"
else:
    import pathlib
    REPO_BASE = str(pathlib.Path("/Users/takehirosato/Desktop/AI_TradeManagement"))

print(f"REPO_BASE={REPO_BASE}")
sys.path.insert(0, f"{REPO_BASE}/scripts")

In [ ]:
# ── 依存パッケージのインストール ──────────────────────────────────────────────
!pip install -q requests anthropic sentence-transformers faiss-cpu

In [ ]:
# ── 環境変数の設定 ────────────────────────────────────────────────────────────
import os

# Colab Secrets または手動設定
# Google Colab: 左サイドバーの「シークレット」から ANTHROPIC_API_KEY を設定
try:
    from google.colab import userdata
    ANTHROPIC_API_KEY = userdata.get("ANTHROPIC_API_KEY")
    BIS_API_KEY       = userdata.get("BIS_API_KEY") or "DEMO_KEY"
except Exception:
    ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY", "")
    BIS_API_KEY       = os.environ.get("BIS_API_KEY", "DEMO_KEY")

if not ANTHROPIC_API_KEY:
    print("⚠️  ANTHROPIC_API_KEY が未設定。Claude API 補完機能は利用できません。")
else:
    print("✅ ANTHROPIC_API_KEY: OK")

print(f"✅ BIS_API_KEY: {'DEMO_KEY' if BIS_API_KEY == 'DEMO_KEY' else '***'}")

In [ ]:
# ── パス設定 ──────────────────────────────────────────────────────────────────
from pathlib import Path

BASE           = Path(REPO_BASE)
DATA_DIR       = BASE / "data"
STAGING_DIR    = DATA_DIR / "staging"
SOURCE_DIR     = DATA_DIR / "source"
UNIFIED_DIR    = DATA_DIR / "unified"

# staging サブディレクトリを確保
for sub in ("sanctions", "fefta", "eccn", "patents", "mappings", "faiss"):
    (STAGING_DIR / sub).mkdir(parents=True, exist_ok=True)

print("パス設定完了:")
for name, p in [("BASE", BASE), ("STAGING_DIR", STAGING_DIR), ("UNIFIED_DIR", UNIFIED_DIR)]:
    print(f"  {name}: {p}  ({'OK' if p.exists() else 'NOT FOUND'})")

## ✅ セットアップ完了
次のノートブック → `01_collect_raw_data.ipynb`